### XGBoost Trees for Regression — Regularized Gradient Boosting

**Building one tree**

0. **Baseline prediction** — a single starting value for every example (default: mean of `y`). All trees after this correct residuals from the baseline.

1. **Single leaf, all residuals**: put every residual $r_i = y_i - \hat{y}_i$ in one leaf and compute

$$\text{Similarity Score} = \frac{\left(\sum r_i\right)^2}{N + \lambda}$$

where $N$ = number of residuals in the leaf, $\lambda$ = regularization parameter.

2. **Why split at all?** One leaf applies the same correction to every residual. Splitting on a feature clusters similar residuals together, so each leaf's correction fits its examples better than one blanket leaf could.

3. **Evaluate a candidate split**: compute the similarity score for the resulting left and right leaves, then

$$\text{Gain} = \text{Similarity}_{\text{left}} + \text{Similarity}_{\text{right}} - \text{Similarity}_{\text{root}}$$

4. **Search for the best split**: repeat step 3 across every threshold, for every feature — the split with the highest gain wins. This exhaustive (or histogram-approximated) scan at every node is what makes tree-building the expensive part of training.

5. **Recurse**: apply the same split logic to each resulting leaf, growing the tree until a stopping condition (max depth, min samples) is hit.

6. **Prune, bottom-up**: starting from the deepest split, compute $\text{Gain} - \gamma$.
   - Negative → prune (merge back into a leaf), then check the parent split the same way.
   - Positive → stop pruning there, even if branches below were already removed.

   $\gamma$ (gamma) is the minimum gain required to justify a split — XGBoost's tree-complexity regularizer.

7. **Leaf output values**, after pruning:

$$\text{Output Value} = \frac{\sum r_i}{N + \lambda}$$

No squaring here — squaring only appears in the similarity score (a measure of how "pure" a leaf's residuals are); the output value is just the regularized average correction for that leaf.

**Combining trees into a prediction**

$$\hat{y} = \text{baseline} + \eta \cdot \left(\text{tree}_1 + \text{tree}_2 + \dots\right)$$

$\eta$ (eta / learning rate, default 0.3) shrinks each tree's contribution so no single tree dominates — this incremental correction is what makes it *boosting*, not averaging.

**Regularization parameters**

- **$\lambda$ (lambda)** — shrinks both the similarity score and the leaf output value. Larger $\lambda$ → smaller output values → more splits fail the $\text{Gain} - \gamma$ pruning test → simpler trees. This is XGBoost's L2 regularization on leaf weights.
- **$\gamma$ (gamma)** — the minimum loss reduction a split must produce to survive pruning. Controls tree size/complexity directly, independent of $\lambda$.


### XGBoost Trees for Classification — Same Mechanics, Different Loss

Everything from the regression case — similarity score, gain, pruning, output value — works identically here. **Only the inputs to the formulas change**, because classification uses log loss instead of squared error.

**The key substitution: gradients and Hessians**

For binary log loss, at raw score (log-odds) $z$ with predicted probability $p = \text{sigmoid}(z)$:

- **Gradient**: $g_i = p_i - y_i$ → the "residual" is $-g_i = y_i - p_i$
- **Hessian**: $h_i = p_i (1 - p_i)$

This is exactly what "Sum of Previous_Prob × (1 − Previous_Prob)" in the original note meant — **that sum of Hessians stands in for $N$** in the regression formula. Regression is the special case where every Hessian equals 1 (squared error's second derivative is constant), which is why regression's denominator is literally "Number of Residuals."

**Unified formula (covers both regression and classification)**

$$\text{Similarity Score} = \frac{\left(\sum g_i\right)^2}{\sum h_i + \lambda}, \qquad \text{Output Value} = \frac{\sum g_i}{\sum h_i + \lambda}$$

(Sign convention note: XGBoost's own docs write this with $g_i = p_i - y_i$ directly, so you'll sometimes see $-\sum g_i$ on top instead of "sum of residuals" — same quantity, opposite sign convention. Don't let that trip you up comparing against the original paper.)

Gain, pruning, and the split-search process are unchanged from the regression case.

**Building the prediction**

$$z = \text{log-odds}(\text{baseline}) + \eta \cdot \left(\text{tree}_1 + \text{tree}_2 + \dots\right)$$

$$p = \text{sigmoid}(z) = \frac{1}{1 + e^{-z}}$$

⚠️ **Correction from the original note**: the logistic conversion is $\dfrac{1}{1+e^{-z}}$, not $\dfrac{e^{-z}}{1+e^{-z}}$ — that second form actually computes $1-p$, the *inverted* probability (multiply top and bottom by $e^z$ to see it reduces to $\text{sigmoid}(-z)$). Easy sign slip, worth double-checking anytime this is written from memory — it's the same `sigmoid` function used in the from-scratch exercise below.

**Regularization parameters**

- **$\lambda$, $\gamma$** — same role as regression: $\lambda$ shrinks similarity/output values, $\gamma$ sets the minimum gain required to survive pruning.
- **Cover** — the original note's derivation is correct: $\text{Cover} = (\text{denominator of similarity score}) - \lambda = \sum h_i$. `min_child_weight` sets a minimum threshold on Cover for a split to be allowed.
  - **Regression**: $h_i = 1$ always, so Cover literally equals the residual count — "minimum number of residuals per leaf" is an exact description.
  - **Classification**: $h_i = p_i(1-p_i)$ varies per example — largest (0.25) when the model is maximally uncertain at $p=0.5$, shrinking toward 0 as the model becomes confident. So Cover is a **confidence-weighted effective count**, not a literal row count: a leaf can satisfy `min_child_weight` with fewer examples if the model is uncertain about them, or need more examples if it's already confident. Worth stating explicitly if asked to explain `min_child_weight` for a classifier specifically.


In [ ]:
import numpy as np

def sigmoid(z):
    # TODO: implement sigmoid(z) = 1 / (1 + e^-z)
    
    pass

def log_loss(y, p, eps=1e-15):
    # TODO: implement -[y*log(p) + (1-y)*log(1-p)], averaged over all examples
    # clip p to [eps, 1-eps] first to avoid log(0)
    pass

def gradient(y, p):
    # TODO: return p - y  (per-example gradient)
    pass

def hessian(p):
    # TODO: return p * (1 - p)  (per-example Hessian)
    pass


# --- Verify against a known case ---
np.random.seed(0)
y = np.array([1, 0, 1, 1, 0])
z = np.array([0.5, -1.2, 2.0, 0.1, -0.3])  # raw scores, not probabilities

p = sigmoid(z)
print("p:", p)
print("log loss:", log_loss(y, p))
print("gradient:", gradient(y, p))
print("hessian:", hessian(p))

# Cross-check log loss against sklearn
from sklearn.metrics import log_loss as sk_log_loss
print("sklearn log loss:", sk_log_loss(y, p))
